# Ophthalmic disease detection - ConvNeXt-Tiny (final project)

Runs the whole project on Google Colab: get the code, install, locate ODIR-5K, verify it, train, evaluate, plot, predict, Grad-CAM.

**Runtime -> Change runtime type -> GPU is optional.** Training reads a cached frozen stem, so a CPU runtime finishes in about an hour; a GPU is not required.

The notebook only calls the scripts in `scripts/`. Every setting lives in `configs/config.yaml`.

## 1. Get the project

In [ ]:
import os, subprocess, sys, glob, zipfile
from pathlib import Path

REPO_URL = "https://github.com/GargeeTripathi/ophthalmic-screening-research.git"   # git URL of the repository (a private repository needs a token or an uploaded zip)
SUBDIR = "FINAL_PROJECT"       # folder inside the repository that holds this project ("" if it is the root)
PROJECT = Path("/content/project")

def has_project(p): return (Path(p) / "scripts" / "train.py").is_file() and (Path(p) / "configs" / "config.yaml").is_file()

if not has_project(PROJECT):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/_repo"], check=False)
        candidate = Path("/content/_repo") / SUBDIR
        if has_project(candidate):
            PROJECT = candidate
    if not has_project(PROJECT):
        # fall back to a copy in Google Drive, then to an uploaded zip
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as exc:
            print("Drive not mounted:", exc)
        found = [Path(p).parents[1] for p in glob.glob("/content/drive/MyDrive/**/scripts/verify_project.py", recursive=True)]
        if found:
            PROJECT = found[0]
        else:
            from google.colab import files
            print("Upload a zip of the FINAL_PROJECT folder")
            name = next(iter(files.upload()))
            zipfile.ZipFile(name).extractall("/content/_zip")
            PROJECT = [Path(p).parents[1] for p in glob.glob("/content/_zip/**/scripts/verify_project.py", recursive=True)][0]

assert has_project(PROJECT), "project not found"
os.chdir(PROJECT)
print("project:", PROJECT)

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU is fine)")

## 3. Locate the dataset

ODIR-5K is not in the repository (licence). The cell looks for the training JPEGs in Google Drive, and otherwise downloads the public Kaggle copy (about 1.6 GB).

In [ ]:
import pandas as pd
wanted = set(pd.read_csv("data/manifests/splits_patient_disjoint_dedup.csv").original_filename)
sample = list(wanted)[:200]

def images_folder(root):
    root = Path(root)
    if not root.is_dir():
        return None
    best, score = None, 0
    for folder in [root] + [p for p in root.rglob("*") if p.is_dir()]:
        hits = sum((folder / n).is_file() for n in sample)
        if hits > score:
            best, score = folder, hits
    return best if score > 150 else None

found = images_folder(os.environ.get("ODIR_IMAGES", "")) if os.environ.get("ODIR_IMAGES") else None
if found is None and Path("/content/drive/MyDrive").is_dir():
    for candidate in sorted(glob.glob("/content/drive/MyDrive/*ODIR*") + glob.glob("/content/drive/MyDrive/*/*ODIR*")
                            + glob.glob("/content/drive/MyDrive/*Dataset*")):
        if candidate.endswith(".zip"):
            zipfile.ZipFile(candidate).extractall("/content/_odir")
            found = images_folder("/content/_odir")
        else:
            found = images_folder(candidate)
        if found:
            break
if found is None:
    import kagglehub
    found = images_folder(kagglehub.dataset_download("andrewmvd/ocular-disease-recognition-odir5k"))
assert found is not None, "ODIR-5K images not found - put them in Drive or set ODIR_IMAGES"
os.environ["ODIR_IMAGES"] = str(found)
print("images:", found)

## 4. Verify the dataset and the split

In [ ]:
sys.path.insert(0, ".")
from src.data.splits import load_split, leakage_report
frame = load_split()
missing = [p for p in frame.image_path if not Path(p).is_file()]
print(leakage_report(frame))
print("class counts:", frame.label.value_counts().to_dict())
print("missing image files:", len(missing))
assert not missing and leakage_report(frame)["patients_in_more_than_one_fold"] == 0

## 5. Train

Five folds. `--skip-completed` resumes after a disconnect. Use `--folds 0` to train only the deployed model.

In [ ]:
!python scripts/train.py --skip-completed

## 6. Evaluate

In [ ]:
!python scripts/evaluate.py
from IPython.display import Markdown, display
display(Markdown(open("results/final_results.md", encoding="utf-8").read()))

## 7. Plots

In [ ]:
from IPython.display import Image as show
for figure in ["results/training/training_history.png", "results/confusion_matrices/confusion_matrix.png",
               "results/roc/roc_curve.png", "results/pr/precision_recall_curve.png"]:
    display(show(figure, width=760))

## 8. Inference on one image

In [ ]:
example = frame[frame.fold == 0].iloc[0]
print("true label:", example.label)
!python scripts/predict.py --image "{example.image_path}" --gradcam results/gradcam/example_prediction.png
display(show("results/gradcam/example_prediction.png", width=300))

## 9. Grad-CAM examples and project verification

In [ ]:
!python scripts/gradcam.py
display(show("results/gradcam/gradcam_overview.png", width=1000))
!python scripts/verify_project.py